# EP2 - BERTimbau **large** no ColabTreina o `neuralmind/bert-large-portuguese-cased` com o **mesmo protocolo** usadolocalmente, para que os números sejam diretamente comparáveis:- holdout 20% estratificado, `random_state=42`, **intocado**;- validação 10% tirada do dev, para early stopping pelo F1-macro;- avaliação única no holdout.**Antes de rodar:** menu `Ambiente de execução > Alterar tipo de ambiente de execução > GPU`.Saídas para baixar no fim: `probs_large_holdout.npz` (para o ensemble) e`result_large.csv`.

In [ ]:
!nvidia-smi
!pip install -q transformers accelerate openpyxl

## 1. Upload do `train.xlsx`Rode a célula e selecione o arquivo (7,6 MB).

In [ ]:
from google.colab import files
up = files.upload()   # selecione train.xlsx

## 2. Dados e split (idêntico ao local)

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split

LABELS = ["c1", "c234", "c5"]
LABEL_MAP = {l: i for i, l in enumerate(LABELS)}

df = pd.read_excel("train.xlsx").dropna(subset=["resp_text", "clarity"])
df["resp_text"] = df["resp_text"].astype(str)
df["clarity"] = df["clarity"].astype(str).str.strip()
df = df[df["clarity"].isin(LABELS)].copy()
df["label"] = df["clarity"].map(LABEL_MAP)

# MESMO holdout: 20% estratificado, seed 42
dev_df, hold_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
tr, va = train_test_split(dev_df, test_size=0.1, stratify=dev_df["label"], random_state=42)
print(f"treino={len(tr)} val={len(va)} holdout={len(hold_df)}")
print(df["clarity"].value_counts(normalize=True).round(4).to_string())

## 3. Treino`lr=1e-5` porque BERT-large é instável em fine-tuning (lr alto colapsa).`fp16` acelera bastante na T4. Padding dinâmico por batch.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments)

MODEL = "neuralmind/bert-large-portuguese-cased"
MAX_LEN = 384
tokenizer = AutoTokenizer.from_pretrained(MODEL)

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels) if labels is not None else None
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item
    def __len__(self):
        return len(self.enc["input_ids"])

def compute_metrics(p):
    return {"f1_macro": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}

# bf16 e mais estavel que fp16 para BERT-large (fp16 pode divergir / dar NaN).
# A100 e L4 suportam bf16; a T4 so fp16 -> usamos fp16 com um aviso.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16={USE_BF16} fp16={not USE_BF16}")
if not USE_BF16:
    print("AVISO: em fp16, se a perda virar NaN ou a acuracia travar em ~0.33,")
    print("       o treino divergiu. Reduza lr para 5e-6 e rode de novo.")

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
args = TrainingArguments(
    output_dir="./out_large",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=1e-5,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    seed=42,
    report_to="none",
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=DS(tr["resp_text"], tr["label"]),
    eval_dataset=DS(va["resp_text"], va["label"]),
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

## 4. Avaliação no holdout + salvar probabilidades

In [ ]:
hold_labels = hold_df["label"].to_numpy()
pred = trainer.predict(DS(hold_df["resp_text"], hold_df["label"]))
probs = torch.softmax(torch.tensor(pred.predictions.astype("float32")), dim=-1).numpy()
y = probs.argmax(1)

acc = accuracy_score(hold_labels, y)
f1m = f1_score(hold_labels, y, average="macro")
print(f"=== BERTimbau large no holdout ===")
print(f"acuracia={acc:.4f}  f1_macro={f1m:.4f}")
print("\nReferencias locais (mesmo holdout):")
print("  BERT base, 1 semente : 0.4500")
print("  BERT base, 3 sementes: 0.4573")
print("  BERT base + CNB      : 0.4760")
print()
print(classification_report(hold_labels, y, target_names=LABELS, digits=4))
print(pd.DataFrame(confusion_matrix(hold_labels, y), index=LABELS, columns=LABELS))

np.savez("probs_large_holdout.npz", y=hold_labels, probs=probs)
pd.DataFrame([{"tag":"large","model":MODEL,"lr":1e-5,"epochs":3,"acc":acc,"f1_macro":f1m}]
            ).to_csv("result_large.csv", index=False)

## 5. Baixar os resultadosMande esses dois arquivos para o grupo — eu ploto no ensemble e comparo.

In [ ]:
files.download("probs_large_holdout.npz")
files.download("result_large.csv")

---## 6. (SÓ SE o large ganhar) Modelo final para a entregaRode daqui para baixo **apenas se** a acurácia acima superar 0,4760. Aqui o modeloé re-treinado em **todo** o train e gera as probabilidades de `test1` e `test2`.Faça upload de `test1.xlsx` e `test2.xlsx` primeiro.

In [ ]:
up2 = files.upload()   # selecione test1.xlsx e test2.xlsx

In [ ]:
# re-treina em TODO o train (5% apenas para early stopping)
tr2, va2 = train_test_split(df, test_size=0.05, stratify=df["label"], random_state=42)
model2 = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
args2 = TrainingArguments(**{**args.to_dict(), "output_dir": "./out_large_final"})
trainer2 = Trainer(
    model=model2, args=args2,
    train_dataset=DS(tr2["resp_text"], tr2["label"]),
    eval_dataset=DS(va2["resp_text"], va2["label"]),
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer2.train()

for name in ["test1", "test2"]:
    t = pd.read_excel(f"{name}.xlsx")
    t["resp_text"] = t["resp_text"].fillna("").astype(str)
    pr = trainer2.predict(DS(t["resp_text"]))
    pb = torch.softmax(torch.tensor(pr.predictions.astype("float32")), dim=-1).numpy()
    np.savez(f"probs_large_{name}.npz", probs=pb)
    print(f"{name}: {len(t)} linhas | distribuicao predita:")
    print(pd.Series([LABELS[i] for i in pb.argmax(1)]).value_counts(normalize=True).round(4).to_string())
    files.download(f"probs_large_{name}.npz")